# Actin fiber edge detection experiments

The phallodin dye stains the actin fibers inside a cell. We want to address the challenge of detecting these fibers computationally. This notebook contains various filters being applied to the image for experimentation. 

## Load image

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/251205_MonoCa9_Phalloidin_Cellbrite_LowConf/Ca922_Mono_LowConf_Phal_Cellbrite_02_CY5, FITC, DAPI, BF"
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1024,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[5], file_pattern[2], verbose=True)

cy5_container = ImageContainer(files[0],config)
cy5_image = cy5_container.merge()
fitc_container = ImageContainer(files[1],config)
fitc_image = fitc_container.merge()
# cy5_dapi = ImageContainer([files[0],files[2]],config)

Display the images:

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

def invert_image_colors(image: np.ndarray) -> np.ndarray:
    """
    Performs a color inversion on the input image.

    This function inverts the pixel values of an image. For an 8-bit image,
    a pixel with value `p` will become `255 - p`. For a 16-bit image,
    a pixel with value `p` will become `65535 - p`, and so on.
    It works for both grayscale and multi-channel (e.g., RGB) images.
    
    For logical matrices (boolean arrays), 0 (False) becomes 1 (True) and 
    1 (True) becomes 0 (False).

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale (2D) or color (3D).

    Returns:
        np.ndarray: The color-inverted image.
    """
    if not isinstance(image, np.ndarray):
        raise TypeError("Input 'image' must be a NumPy array.")

    # Handle logical matrices (boolean arrays)
    if image.dtype == bool:
        return np.logical_not(image)

    # cv2.bitwise_not performs a bitwise NOT operation on each pixel.
    # For an 8-bit unsigned integer (uint8), this effectively inverts the color.
    # E.g., 0 (binary 00000000) becomes 255 (binary 11111111),
    # and 255 becomes 0.
    inverted_image = cv2.bitwise_not(image)

    return inverted_image

# fig,axes = plt.subplots(1,3,figsize=(16,7))
# cy5_container.display(ax=axes[0],title='Phalloidin')
# cy5_dapi.display(ax=axes[1],title='Phalloidin+DAPI')
# axes[2].imshow(invert_image_colors(cy5_image))
# axes[2].axis('off')
# axes[2].set_title('Inverted colors')

# plt.show()

## Helper functions

### Otsu's thresholding

In [ ]:
import cv2
import numpy as np

def get_otsu_edge_map(image: np.ndarray) -> np.ndarray:
    """
    Applies Otsu's thresholding to an input array (e.g., an edge magnitude map)
    to return a binary edge map.

    Args:
        image (np.ndarray): Input array. Can be float or integer type.

    Returns:
        np.ndarray: Binary map (0s and 1s) where 1 represents the edges/foreground.
    """
    # OpenCV's Otsu implementation requires an 8-bit single-channel input image.
    if image.dtype != np.uint8:
        # Normalize the image to the 0-255 range for 8-bit conversion
        min_val = image.min()
        max_val = image.max()
        
        if max_val > min_val:
            # Scale to 0-255
            img_8u = (255 * (image - min_val) / (max_val - min_val)).astype(np.uint8)
        else:
            # Handle constant image case
            img_8u = np.zeros_like(image, dtype=np.uint8)
    else:
        img_8u = image

    # Apply Otsu's thresholding
    # cv2.THRESH_OTSU tells OpenCV to calculate the optimal threshold.
    # The explicit threshold value (0) is ignored.
    # maxval is set to 1 so the resulting binary map contains 0s and 1s.
    otsu_thresh_val, binary_map = cv2.threshold(
        img_8u, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    
    return binary_map


### Canny style post-processing

In [ ]:
import numpy as np
import cv2

def apply_canny_post_processing(edge_magnitude: np.ndarray, gx: np.ndarray, gy: np.ndarray, min_size: int = 15) -> np.ndarray:
    """
    Applies Canny-style post-processing to an edge magnitude map.
    
    Steps:
    1. Non-Maximum Suppression (NMS) using gradient orientation to thin edges.
    2. Double Thresholding (Hysteresis) using Otsu's method for the high threshold.
    3. Small edge removal (filtering components smaller than min_size).
    
    Args:
        edge_magnitude (np.ndarray): The magnitude of the edges.
        gx (np.ndarray): Gradient in x direction.
        gy (np.ndarray): Gradient in y direction.
        min_size (int): Minimum size (in pixels) for an edge segment to be kept.
        
    Returns:
        np.ndarray: Binary edge map (0s and 1s).
    """
    # 1. Calculate Gradient Orientation
    # Result is in degrees, range [-180, 180]
    orientation = np.arctan2(gy, gx) * (180 / np.pi)
    orientation[orientation < 0] += 180 # Map to [0, 180]
    
    # 2. Non-Maximum Suppression (NMS)
    H, W = edge_magnitude.shape
    nms_map = np.zeros((H, W), dtype=np.float32)
    
    # Pad for vectorized neighbor access
    M_pad = np.pad(edge_magnitude, 1, mode='constant')
    
    # Define masks for the 4 orientation bins
    # 0 degrees (Horizontal gradient, Vertical Edge): Check Left/Right
    mask_0 = np.logical_or(orientation < 22.5, orientation >= 157.5)
    # 45 degrees (Diagonal gradient): Check TopRight/BottomLeft
    mask_45 = np.logical_and(orientation >= 22.5, orientation < 67.5)
    # 90 degrees (Vertical gradient, Horizontal Edge): Check Top/Bottom
    mask_90 = np.logical_and(orientation >= 67.5, orientation < 112.5)
    # 135 degrees (Diagonal gradient): Check TopLeft/BottomRight
    mask_135 = np.logical_and(orientation >= 112.5, orientation < 157.5)
    
    # Vectorized NMS comparison
    # Note: M_pad indices are shifted by +1 compared to original
    
    # 0 deg: Neighbors (i, j+1) -> pad(i+1, j+2) and (i, j-1) -> pad(i+1, j)
    # Slicing M_pad[1:-1, :] gives rows 1 to H.
    q0 = M_pad[1:-1, 2:]   # Right neighbor
    r0 = M_pad[1:-1, :-2]  # Left neighbor
    nms_map[mask_0] = np.where((edge_magnitude[mask_0] >= q0[mask_0]) & 
                               (edge_magnitude[mask_0] >= r0[mask_0]), 
                               edge_magnitude[mask_0], 0)
    
    # 45 deg: Neighbors (i-1, j+1) -> pad(i, j+2) and (i+1, j-1) -> pad(i+2, j)
    q45 = M_pad[:-2, 2:]   # Top-Right
    r45 = M_pad[2:, :-2]   # Bottom-Left
    nms_map[mask_45] = np.where((edge_magnitude[mask_45] >= q45[mask_45]) & 
                                (edge_magnitude[mask_45] >= r45[mask_45]), 
                                edge_magnitude[mask_45], 0)

    # 90 deg: Neighbors (i-1, j) -> pad(i, j+1) and (i+1, j) -> pad(i+2, j+1)
    q90 = M_pad[:-2, 1:-1] # Top
    r90 = M_pad[2:, 1:-1]  # Bottom
    nms_map[mask_90] = np.where((edge_magnitude[mask_90] >= q90[mask_90]) & 
                                (edge_magnitude[mask_90] >= r90[mask_90]), 
                                edge_magnitude[mask_90], 0)

    # 135 deg: Neighbors (i-1, j-1) -> pad(i, j) and (i+1, j+1) -> pad(i+2, j+2)
    q135 = M_pad[:-2, :-2] # Top-Left
    r135 = M_pad[2:, 2:]   # Bottom-Right
    nms_map[mask_135] = np.where((edge_magnitude[mask_135] >= q135[mask_135]) & 
                                 (edge_magnitude[mask_135] >= r135[mask_135]), 
                                 edge_magnitude[mask_135], 0)

    # 3. Double Thresholding (Hysteresis)
    # Use Otsu's method to determine the High Threshold automatically
    if nms_map.max() > 0:
        # Normalize to 0-255 for Otsu
        nms_8u = (nms_map / nms_map.max() * 255).astype(np.uint8)
        otsu_thresh_val, _ = cv2.threshold(nms_8u, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        high_thresh = otsu_thresh_val / 255.0 * nms_map.max()
    else:
        high_thresh = 0
        
    low_thresh = high_thresh * 0.5
    
    # Identify Strong and Weak edges
    strong_mask = (nms_map >= high_thresh).astype(np.uint8)
    weak_mask = ((nms_map >= low_thresh) & (nms_map < high_thresh)).astype(np.uint8)
    
    # 4. Connectivity Analysis & Size Filtering
    # Combine strong and weak edges to find connected components
    candidates = strong_mask + weak_mask
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(candidates, connectivity=8)
    
    final_map = np.zeros_like(edge_magnitude, dtype=np.uint8)
    
    # Iterate through components (skip label 0 which is background)
    for i in range(1, num_labels):
        # Size Filtering: Remove small edges
        if stats[i, cv2.CC_STAT_AREA] < min_size:
            continue
            
        # Hysteresis: Keep component only if it contains at least one strong pixel
        # Create a mask for the current component
        component_mask = (labels == i)
        
        # Check intersection with strong edges
        if np.any(strong_mask[component_mask]):
            final_map[component_mask] = 1
            
    return final_map

## Kernel Convolution

Kernel convolution computes the weighted sum of the pixels around one pixel. Depending on the kernel used, the effect is different.

### Simple Kernel Convolutions

Run one kernel convolution at a time

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data # Assuming skimage is available for a sample image

def convolve_with_multiple_kernels(
    image: np.ndarray,
    kernels: dict,
    figsize: tuple = (12, 16),
    cmap: str = 'gray'
):
    """
    Applies multiple 2D convolutions to an image using a dictionary of kernels
    and displays all results for comparison.

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale (2D) or color (3D).
        kernels (dict): A dictionary where keys are kernel names (str) and
                        values are the 2D kernel arrays (np.ndarray).
        figsize (tuple): The size of the visualization figure.
        cmap (str): The colormap to use for displaying grayscale images.
    """
    if not isinstance(image, np.ndarray) or not isinstance(kernels, dict):
        raise TypeError("Input 'image' must be a NumPy array and 'kernels' must be a dictionary.")

    # Determine the grid size for the plot
    num_kernels = len(kernels)
    num_plots = num_kernels + 1  # +1 for the original image
    
    # Arrange plots in a grid, with a maximum of 3 columns
    ncols = min(3, num_plots)
    nrows = int(np.ceil(num_plots / ncols))
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.ravel()  # Flatten the axes array for easy iteration

    # --- 1. Display the Original Image ---
    axes[0].imshow(image, cmap=cmap if image.ndim == 2 else None)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    # --- 2. Apply and Display Each Convolution ---
    for i, (kernel_name, kernel) in enumerate(kernels.items()):
        ax = axes[i + 1]
        
        # Apply the convolution using cv2.filter2D
        # ddepth=-1 means the output image will have the same bit-depth as the input.
        convolved_image = cv2.filter2D(image, -1, kernel)
        
        ax.imshow(convolved_image, cmap=cmap if convolved_image.ndim == 2 else None)
        ax.set_title(kernel_name)
        ax.axis('off')

    # --- 3. Clean up unused subplots ---
    for i in range(num_plots, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


# --- Example Usage ---

# Assuming 'cy5_image' is your loaded image. For demonstration, we'll use a sample image.
try:
    # Check if cy5_image exists, otherwise use a sample image
    cy5_image
except NameError:
    print("`cy5_image` not found. Using a sample image from skimage.data for demonstration.")
    cy5_image = data.camera()


# Define the kernels to be applied
ridge_kernel = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]])
edge_kernel = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])
sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
sobel_x_kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]) # Sobel X is a better name
negative_sobel_x_kernel = -sobel_x_kernel
sobel_y_kernel = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]) # Sobel Y
negative_sobel_y_kernel = -sobel_y_kernel
# Corrected syntax for the Gaussian kernel
gaussian_kernel = (1/16) * np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])

# Create a dictionary of kernels for the new function
all_kernels = {
    "Gaussian Blur": gaussian_kernel,
    "Sharpen": sharpen_kernel,
    "Edge Detection": edge_kernel,
    "Ridge Detection": ridge_kernel,
    "Sobel X (Vertical Edges)": sobel_x_kernel,
    "Negative Sobel X": negative_sobel_x_kernel,
    "Sobel Y (Horizontal Edges)": sobel_y_kernel,
    "Negative Sobel Y": negative_sobel_y_kernel,
}

# Call the new function to display all results
convolve_with_multiple_kernels(cy5_image, all_kernels)

For a better visualized example, we run it on an example image

In [ ]:
from skimage import data
convolve_with_multiple_kernels(data.camera(), all_kernels)

## Edge Preserving Smoothing

The Gaussian kernel won't distinguish between edges and non-edges. There are edge-preserving smoothing methods: median smoothing, bilaterial smoothing, and sigma smoothing.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.util import random_noise

def apply_edge_preserving_smoothing(image: np.ndarray, method: str = 'median', **kwargs):
    """
    Applies a smoothing filter to an image using OpenCV.

    Args:
        image (np.ndarray): 
            The input image (should be 8-bit, e.g., uint8).
        method (str): 
            The smoothing method to use. Can be 'median', 'bilateral', or 'gaussian'.
        **kwargs: 
            Additional arguments for the chosen filter function.
            
            For 'median':
                ksize (int): Aperture linear size; must be odd and greater than 1.
            
            For 'bilateral':
                d (int): Diameter of each pixel neighborhood.
                sigmaColor (float): Filter sigma in the color space.
                sigmaSpace (float): Filter sigma in the coordinate space.

            For 'gaussian':
                ksize (tuple): Kernel size, e.g., (5, 5). Must be positive and odd.
                sigmaX (float): Gaussian kernel standard deviation in X direction.

    Returns:
        np.ndarray: The smoothed image.
    """
    if method.lower() == 'median':
        ksize = kwargs.get('ksize', 5)
        if ksize % 2 == 0 or ksize <= 1:
            raise ValueError("ksize for median filter must be an odd integer greater than 1.")
        return cv2.medianBlur(image, ksize)
    
    elif method.lower() == 'bilateral':
        d = kwargs.get('d', 9)
        sigma_color = kwargs.get('sigmaColor', 75)
        sigma_space = kwargs.get('sigmaSpace', 75)
        return cv2.bilateralFilter(image, d, sigma_color, sigma_space)

    elif method.lower() == 'gaussian':
        ksize = kwargs.get('ksize', (5, 5))
        if not isinstance(ksize, tuple) or len(ksize) != 2 or ksize[0] % 2 == 0 or ksize[1] % 2 == 0:
            raise ValueError("ksize for Gaussian filter must be a tuple of two odd integers.")
        sigmaX = kwargs.get('sigmaX', 0)
        return cv2.GaussianBlur(image, ksize, sigmaX)
    
    else:
        raise ValueError("Unsupported method. Choose 'median', 'bilateral', or 'gaussian'.")

# --- Example Usage ---

# 1. Load a sample image and convert to 8-bit grayscale
original_image = (data.camera()).astype(np.uint8)

# 2. Add some 'salt & pepper' noise to demonstrate the filter's effectiveness
noisy_image = random_noise(original_image, mode='s&p', amount=0.05)
noisy_image = (noisy_image * 255).astype(np.uint8)

# 3. Apply the filters
# Apply Median Filter (Edge-Preserving)
median_smoothed = apply_edge_preserving_smoothing(noisy_image, method='median', ksize=5)

# Apply Bilateral Filter (Edge-Preserving)
bilateral_smoothed = apply_edge_preserving_smoothing(noisy_image, method='bilateral', d=9, sigmaColor=75, sigmaSpace=75)

# Apply Gaussian Filter (Non-Edge-Preserving)
gaussian_smoothed = apply_edge_preserving_smoothing(noisy_image, method='gaussian', ksize=(5, 5))

# 4. Display the results for comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(original_image, cmap='gray')
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_image, cmap='gray')
axes[0, 1].set_title("Image with Salt & Pepper Noise")
axes[0, 1].axis('off')

axes[0, 2].imshow(gaussian_smoothed, cmap='gray')
axes[0, 2].set_title("Gaussian Smoothing (Blurs Edges)")
axes[0, 2].axis('off')

axes[1, 0].imshow(median_smoothed, cmap='gray')
axes[1, 0].set_title("Median Filter (Edge-Preserving)")
axes[1, 0].axis('off')

axes[1, 1].imshow(bilateral_smoothed, cmap='gray')
axes[1, 1].set_title("Bilateral Filter (Edge-Preserving)")
axes[1, 1].axis('off')

# Hide the unused subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Discrete Wavelet Transform

First list wavelet families

In [ ]:
import pywt

# List all available wavelet families
print("Available wavelet families:")
print(pywt.families(short=False))
print(pywt.families(short=True))
print(pywt.wavelist(kind='discrete'))

# List all wavelets within the Daubechies family
print("\nWavelets in the Daubechies family:")
print(pywt.wavelist(family='gaus'))

### Denoising with wavelet transform

Wavelets to try: Haar, Daubechies, Symlets, Coiflets. These are orthogonal wavelets.

In [ ]:
import pywt
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.util import random_noise

def denoise_image_2d(image, wavelet='db1', level=None, threshold_mode='soft', threshold_value=None):
    """
    Denoises a 2D image (H x W numpy array) using Wavelet Transform.
    
    Args:
        image (np.ndarray): Input 2D image.
        wavelet (str): Wavelet name (e.g., 'db1', 'sym2').
        level (int): Decomposition level. If None, calculated automatically.
        threshold_mode (str): Thresholding mode ('soft' or 'hard').
        threshold_value (float): Threshold value. If None, estimated using VisuShrink.
        
    Returns:
        np.ndarray: Denoised image.
    """
    # Perform multi-level wavelet decomposition
    # coeffs is a list: [cAn, (cHn, cVn, cDn), ..., (cH1, cV1, cD1)]
    coeffs = pywt.wavedec2(image, wavelet, level=level)
    
    # Estimate threshold if not provided
    if threshold_value is None:
        # VisuShrink estimation: sigma * sqrt(2 * log(N))
        # Estimate noise sigma from the finest detail coefficients (diagonal of the first level)
        # coeffs[-1] corresponds to level 1 details: (cH1, cV1, cD1)
        cD1 = coeffs[-1][-1] # diagonal detail coefficient
        # Median absolute deviation estimator: estimates noise variance
        sigma = np.median(np.abs(cD1)) / 0.6745
        # calculates the threshold based on the size of the image
        threshold_value = sigma * np.sqrt(2 * np.log(image.size))
    
    # Apply thresholding to detail coefficients
    # We typically keep the approximation coefficients (coeffs[0]) unchanged
    coeffs_thresholded = list(coeffs)
    for i in range(1, len(coeffs)):
        coeffs_thresholded[i] = tuple(
            pywt.threshold(c, threshold_value, mode=threshold_mode) for c in coeffs[i]
        )
        
    # Reconstruct the image
    denoised_image = pywt.waverec2(coeffs_thresholded, wavelet)
    
    # Ensure the output shape matches the input (reconstruction can sometimes add padding)
    if denoised_image.shape != image.shape:
        denoised_image = denoised_image[:image.shape[0], :image.shape[1]]
        
    if np.issubdtype(image.dtype, np.integer):
        info = np.iinfo(image.dtype)
        min_val = denoised_image.min()
        max_val = denoised_image.max()
        
        if max_val > min_val:
            # Scale to [0, info.max] (e.g., 0-255 or 0-65535)
            denoised_image = (denoised_image - min_val) / (max_val - min_val) * info.max
        else:
            # Handle constant image case (avoid division by zero)
            denoised_image = np.zeros_like(denoised_image)
            
        denoised_image = denoised_image.astype(image.dtype)
        
    return denoised_image

# List of wavelets to try
denoise_wavelets = ("db1", # Haar
                    "db4", # Daubechies 4
                    "sym2", # Symlet 2
                    "sym6", # Symlet 6
                    "sym8",
                    "coif1", # Coiflet 1
                    "coif4", # Coiflet 4
                    "coif8",
                    )

# --- Visualization Loop ---
# Assuming 'image' is your H x W numpy array defined in your notebook.
# If testing without an image, uncomment the lines below to create one:
# original = np.zeros((256, 256)); original[64:192, 64:192] = 1.0
# image = original + 0.1 * np.random.randn(*original.shape)
original = data.camera()
image = random_noise(original, mode='s&p', amount=0.05)
image = (image * 255).astype(np.uint8)
# image = cy5_image

plt.figure(figsize=(15, 10))

# Plot the original image
plt.subplot(3, 3, 1)
# plt.imshow(invert_image_colors(image), cmap='gray')
plt.imshow(image, cmap='gray')
plt.title("Original Image")
plt.axis('off')

# Loop through each wavelet and plot the result
for i, wavelet in enumerate(denoise_wavelets):
    # Apply denoising
    # You can pass a specific threshold_value here if the automatic one is too strong/weak
    denoised = denoise_image_2d(image, wavelet=wavelet, level=4)
    
    # Plot result
    plt.subplot(3, 4, i + 2)
    # plt.imshow(invert_image_colors(denoised), cmap='gray')
    plt.imshow(denoised, cmap='gray')
    plt.title(f"Wavelet: {wavelet}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pywt
import cv2
import matplotlib.pyplot as plt
# from image_processing_tools.util.visualize import add_gradient_arrows_to_ax
from matplotlib import colors

def add_gradient_arrows_to_ax(
    ax: plt.Axes,
    gx: np.ndarray,
    gy: np.ndarray,
    step: int = 20,
    arrow_scale: float = 1.0,
    arrow_width: float = 0.005,
    alpha: float = 0.8,
    rotate_90: bool = False,
    show_arrowheads: bool = False,
    uniform_length: bool = True,
    color_by_magnitude: bool = True,
    colormap: str = 'viridis'
):
    """
    Overlays arrows representing gradient direction and magnitude on a matplotlib axis.

    Args:
        ax (plt.Axes): The matplotlib axis to draw on.
        gx (np.ndarray): Gradient in x direction.
        gy (np.ndarray): Gradient in y direction.
        step (int): Spacing between arrows (in pixels).
        arrow_scale (float): Scaling factor for arrow length.
        arrow_width (float): Width of the arrow shaft.
        alpha (float): Transparency of the arrows.
        rotate_90 (bool): If True, rotates the arrows by 90 degrees.
        show_arrowheads (bool): If True, draws arrow heads. Defaults to False (lines only).
        uniform_length (bool): If True, all arrows have the same length.
        color_by_magnitude (bool): If True, colors arrows by gradient magnitude.
        colormap (str): The name of the colormap to use if color_by_magnitude is True.
    """
    h, w = gx.shape
    # Create grid coordinates
    y, x = np.mgrid[step//2:h:step, step//2:w:step]
    
    # Sample gradients
    u = gx[step//2:h:step, step//2:w:step]
    v = gy[step//2:h:step, step//2:w:step]
    
    # Calculate magnitude
    magnitude = np.sqrt(u**2 + v**2)
    
    # Mask out zero gradients to avoid clutter
    mask = magnitude > 0
    if not np.any(mask):
        return

    x = x[mask]
    y = y[mask]
    u = u[mask]
    v = v[mask]
    magnitude = magnitude[mask]
    
    if rotate_90:
        # Rotate vectors (u, v) by 90 degrees: (x, y) -> (-y, x)
        u_rot = -v
        v_rot = u
        u, v = u_rot, v_rot
        
    # Rotate opposing vectors such that they are on the same direction    
    u[(u<0)&(v<0)] = np.abs(u[(u<0)&(v<0)])
    u[(u>0)&(v<0)] = -u[(u>0)&(v<0)]
    
    if color_by_magnitude:
        norm = colors.Normalize(vmin=magnitude.min(), vmax=magnitude.max())
        cmap = plt.get_cmap(colormap)
        arrow_colors = cmap(norm(magnitude))
    else:
        # Calculate angle for coloring (in degrees)
        angle = np.arctan2(v, u) * (180 / np.pi)
        
        # Normalize angle to 0-1 for Hue (0-360 -> 0-1)
        hue = ((angle + 180) % 360) / 360.0
        saturation = np.ones_like(hue)
        value = np.ones_like(hue)
        
        hsv = np.stack((hue, saturation, value), axis=-1)
        arrow_colors = colors.hsv_to_rgb(hsv)
    
    if uniform_length:
        # Normalize vectors to unit length and scale uniformly
        u = u / magnitude
        v = v / magnitude
        fixed_length = step * arrow_scale * 0.8
        u *= fixed_length
        v *= fixed_length
    else:
        # Scale vector length by magnitude. The longest arrow will have a length
        # proportional to `step * arrow_scale`.
        max_magnitude = np.max(magnitude)
        if max_magnitude > 0:
            scaling_factor = (step * arrow_scale * 0.8) / max_magnitude
            u *= scaling_factor
            v *= scaling_factor

    headwidth = 3 if show_arrowheads else 0
    headlength = 5 if show_arrowheads else 0
    headaxislength = 4.5 if show_arrowheads else 0

    ax.quiver(x, y, u, v, color=arrow_colors, angles='xy', scale_units='xy', scale=1, 
              width=arrow_width, pivot='mid', alpha=alpha, 
              headwidth=headwidth, headlength=headlength, headaxislength=headaxislength)

def calculate_gradient_direction_color(gx: np.ndarray, gy: np.ndarray) -> np.ndarray:
    """
    Calculates gradient directions and represents them as a color image (HSV -> RGB).
    Low magnitude areas appear white (low saturation), high magnitude areas appear colorful.
    The direction is rotated by 90 degrees (orthogonal to original).
    
    Args:
        gx (np.ndarray): Gradient in x direction.
        gy (np.ndarray): Gradient in y direction.
        
    Returns:
        np.ndarray: RGB image representing gradient directions.
    """
    # Calculate magnitude and angle
    magnitude = np.sqrt(gx**2 + gy**2)
    angle = np.arctan2(gy, gx) * (180 / np.pi) # Degrees -180 to 180
    
    # Rotate angle by 90 degrees
    angle += 90
    
    # Normalize angle to [0, 360) range
    angle = angle % 360
    
    angle = angle % 180
    
    # Map angle to 0-180 for HSV Hue (OpenCV expects 0-179)
    hue = (angle / 2).astype(np.uint8)
    
    # Value is set to maximum (bright) so the background is white
    value = np.ones_like(hue) * 255
    # value = (magnitude / magnitude.max() * 255).astype(np.uint8)
    
    # Saturation is set based on normalized magnitude
    if magnitude.max() > 0:
        saturation = (magnitude / magnitude.max() * 255).astype(np.uint8)
    else:
        saturation = np.zeros_like(hue)
        
    # Create HSV image
    hsv = cv2.merge([hue, saturation, value])
    
    # Convert to RGB for display
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return rgb


def process_actin_fibers_correct(img, edge_method='mexican_hat', post_processing=False):
    """
    Process actin fibers with denoising, edge detection, and multiscale product.
    
    Args:
        img (np.ndarray): Input 2D image.
        edge_method (str): 'mexican_hat' (2nd derivative), 'gaussian' (1st derivative), or 'haar'.
        post_processing (bool): Whether to apply Canny-style post-processing (NMS + Hysteresis).
    """

    # --- PART 1: DENOISING (Sym8) ---
    wavelet_denoise = 'sym8'
    level = 3
    coeffs = pywt.wavedec2(img, wavelet_denoise, level=level)
    
    sigma = np.median(np.abs(coeffs[-1][-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(img.size))
    
    new_coeffs = [coeffs[0]]
    for i in range(1, len(coeffs)):
        new_coeffs.append(tuple(pywt.threshold(c, threshold, mode='soft') for c in coeffs[i]))
    
    img_denoised = pywt.waverec2(new_coeffs, wavelet_denoise)

    # --- PART 2: EDGE DETECTION ---
    sigma_scale = 1.5
    img_blurred = cv2.GaussianBlur(img_denoised, (0, 0), sigma_scale)
    
    gx, gy, edges = None, None, None
    
    if edge_method == 'mexican_hat':
        edges = cv2.Laplacian(img_blurred, cv2.CV_64F)
        edges = np.abs(edges)
        gx = cv2.Sobel(img_blurred, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img_blurred, cv2.CV_64F, 0, 1, ksize=3)
        
    elif edge_method == 'gaussian':
        gx = cv2.Sobel(img_blurred, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img_blurred, cv2.CV_64F, 0, 1, ksize=3)
        edges = np.sqrt(gx**2 + gy**2)

    elif edge_method == 'haar':
        coeffs = pywt.dwt2(img_denoised, 'haar')
        cA, (cH, cV, cD) = coeffs
        
        h, w = img.shape
        cH = cv2.resize(cH, (w, h), interpolation=cv2.INTER_LINEAR)
        cV = cv2.resize(cV, (w, h), interpolation=cv2.INTER_LINEAR)
        cD = cv2.resize(cD, (w, h), interpolation=cv2.INTER_LINEAR)
        
        edges = np.sqrt(cH**2 + cV**2 + cD**2)
        gx = cH
        gy = cV

    else:
        raise ValueError("edge_method must be 'mexican_hat', 'gaussian', or 'haar'")

    # Calculate gradient directions for visualization
    grad_colors = calculate_gradient_direction_color(gx, gy)

    # Apply post-processing based on the flag
    if post_processing:
        edges_binary = apply_canny_post_processing(edges, gx, gy, min_size=15)
    else:
        edges_binary = get_otsu_edge_map(edges)

    # --- PART 3: MULTISCALE PRODUCT (The Correlation Step) ---
    coeffs_mp = pywt.wavedec2(img_denoised, 'bior3.5', level=level)
    
    def get_mag(level_coeffs):
        LH, HL, HH = level_coeffs
        return np.sqrt(LH**2 + HL**2 + HH**2)

    mag_l1 = get_mag(coeffs_mp[-1])
    mag_l2 = get_mag(coeffs_mp[-2])
    
    mag_l2_resized = cv2.resize(mag_l2, (mag_l1.shape[1], mag_l1.shape[0]), interpolation=cv2.INTER_LANCZOS4)
    
    multiscale_prod = mag_l1 * mag_l2_resized
    
    if np.issubdtype(img.dtype, np.integer):
        info = np.iinfo(img.dtype)
        min_val = multiscale_prod.min()
        max_val = multiscale_prod.max()
        
        if max_val > min_val:
            multiscale_prod = (multiscale_prod - min_val) / (max_val - min_val) * info.max
        else:
            multiscale_prod = np.zeros_like(multiscale_prod)
            
        multiscale_prod = multiscale_prod.astype(img.dtype)
    
    multiscale_prod_binary = get_otsu_edge_map(multiscale_prod)    

    # --- VISUALIZATION ---
    def safe_invert(im):
        try: return invert_image_colors(im)
        except NameError: return 255 - im if im.dtype==np.uint8 else im
        
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
    axes[1].imshow(img_denoised, cmap='gray'); axes[1].set_title('Denoised (Sym8)')
    add_gradient_arrows_to_ax(axes[1], gx, gy, step=50, rotate_90=True, arrow_scale=1, alpha=0.8, colormap='gray')
    axes[2].imshow(grad_colors); axes[2].set_title('Gradient Directions')
    # add_gradient_arrows_to_ax(axes[2], gx, gy, step=50, rotate_90=True, arrow_scale=1, alpha=0.8, colormap='cividis')
    
        
    axes[3].imshow(safe_invert(edges_binary.astype(bool)), cmap='gray', interpolation = 'nearest')
    # add_gradient_arrows_to_ax(axes[2], gx, gy, step=100, rotate_90=True, alpha=0.8, arrow_scale=3)
    axes[3].set_title(f'Edge Detection ({edge_method})')
    
    # axes[4].imshow(safe_invert(multiscale_prod_binary.astype(bool)), cmap='gray', interpolation = 'nearest')
    # axes[4].set_title('Multiscale Product')
    
    for ax in axes:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
                                                                                            
process_actin_fibers_correct(fitc_image, edge_method='gaussian')

In [ ]:
import numpy as np
import pywt
import cv2
import matplotlib.pyplot as plt

def process_actin_fibers_correct(img, edge_method='mexican_hat'):
    """
    Process actin fibers with denoising, edge detection, and multiscale product.
    
    Args:
        img (np.ndarray): Input 2D image.
        edge_method (str): 'mexican_hat' (2nd derivative), 'gaussian' (1st derivative), or 'haar'.
    """

    # --- PART 1: DENOISING (Sym8) ---
    # Using Discrete Wavelet Transform for Denoising
    wavelet_denoise = 'sym8'
    level = 3
    coeffs = pywt.wavedec2(img, wavelet_denoise, level=level)
    
    # Estimate noise from the finest detail level (HH)
    # sigma = Median Absolute Deviation / 0.6745
    sigma = np.median(np.abs(coeffs[-1][-1])) / 0.6745
    threshold = sigma * np.sqrt(2 * np.log(img.size))
    
    # Soft Thresholding
    new_coeffs = [coeffs[0]]
    for i in range(1, len(coeffs)):
        new_coeffs.append(tuple(pywt.threshold(c, threshold, mode='soft') for c in coeffs[i]))
    
    img_denoised = pywt.waverec2(new_coeffs, wavelet_denoise)

    # --- PART 2: EDGE DETECTION ---
    # We use spatial filters which are equivalent to continuous wavelet transforms
    # at a specific scale, or discrete wavelet transforms (Haar).
    sigma_scale = 1.5
    img_blurred = cv2.GaussianBlur(img_denoised, (0, 0), sigma_scale)
    
    gx, gy, edges = None, None, None
    
    if edge_method == 'mexican_hat':
        # Mexican Hat is the 2nd derivative of Gaussian (Laplacian of Gaussian)
        # It highlights ridges and blobs (zero-crossings).
        edges = cv2.Laplacian(img_blurred, cv2.CV_64F)
        edges = np.abs(edges) # Magnitude of the response
        # Calculate gradients for NMS orientation
        gx = cv2.Sobel(img_blurred, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img_blurred, cv2.CV_64F, 0, 1, ksize=3)
        
    elif edge_method == 'gaussian':
        # "Gaussian Wavelet" usually refers to the 1st derivative of Gaussian (Gradient).
        # It highlights step edges.
        gx = cv2.Sobel(img_blurred, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(img_blurred, cv2.CV_64F, 0, 1, ksize=3)
        edges = np.sqrt(gx**2 + gy**2) # Gradient magnitude

    elif edge_method == 'haar':
        # Haar Wavelet Transform
        # Uses the detail coefficients from the first level decomposition to detect edges.
        coeffs = pywt.dwt2(img_denoised, 'haar')
        cA, (cH, cV, cD) = coeffs
        
        # Resize components to original resolution
        h, w = img.shape
        cH = cv2.resize(cH, (w, h), interpolation=cv2.INTER_LINEAR)
        cV = cv2.resize(cV, (w, h), interpolation=cv2.INTER_LINEAR)
        cD = cv2.resize(cD, (w, h), interpolation=cv2.INTER_LINEAR)
        
        edges = np.sqrt(cH**2 + cV**2 + cD**2)
        gx = cH # Horizontal detail corresponds to vertical edges (dx)
        gy = cV # Vertical detail corresponds to horizontal edges (dy)

    else:
        raise ValueError("edge_method must be 'mexican_hat', 'gaussian', or 'haar'")

    # Apply Canny-style post-processing (NMS + Hysteresis + Size Filtering)
    # This returns a binary map (0s and 1s)
    # edges_binary = apply_canny_post_processing(edges, gx, gy, min_size=0)
    edges_binary = get_otsu_edge_map(edges)

    # --- PART 3: MULTISCALE PRODUCT (The Correlation Step) ---
    # We use 'bior3.5' because of its high symmetry and smoothness
    coeffs_mp = pywt.wavedec2(img_denoised, 'bior3.5', level=level)
    
    def get_mag(level_coeffs):
        LH, HL, HH = level_coeffs
        return np.sqrt(LH**2 + HL**2 + HH**2)

    mag_l1 = get_mag(coeffs_mp[-1]) # Finest details
    mag_l2 = get_mag(coeffs_mp[-2]) # Next level up
    
    # Resize Mag_L2 to match Mag_L1 for pixel-wise multiplication
    mag_l2_resized = cv2.resize(mag_l2, (mag_l1.shape[1], mag_l1.shape[0]), interpolation=cv2.INTER_LANCZOS4)
    
    # The Multiscale Product amplifies the structural fibers and suppresses random noise
    multiscale_prod = mag_l1 * mag_l2_resized
    
    # Normalize Multiscale Product
    if np.issubdtype(img.dtype, np.integer):
        info = np.iinfo(img.dtype)
        min_val = multiscale_prod.min()
        max_val = multiscale_prod.max()
        
        if max_val > min_val:
            multiscale_prod = (multiscale_prod - min_val) / (max_val - min_val) * info.max
        else:
            multiscale_prod = np.zeros_like(multiscale_prod)
            
        multiscale_prod = multiscale_prod.astype(img.dtype)
    
    # Apply Otsu's thresholding to get binary multiscale product map
    multiscale_prod_binary = get_otsu_edge_map(multiscale_prod)    

    # --- VISUALIZATION ---
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
    axes[1].imshow(img_denoised, cmap='gray'); axes[1].set_title('Denoised (Sym8)')
    
    # Helper to invert colors if defined, otherwise just show
    def safe_invert(im):
        try: return invert_image_colors(im)
        except NameError: return 255 - im if im.dtype==np.uint8 else im # Fallback
        
    axes[2].imshow(safe_invert(edges_binary.astype(bool)), cmap='gray', interpolation = 'nearest')
    axes[2].set_title(f'Edge Detection ({edge_method})')
    
    axes[3].imshow(safe_invert(multiscale_prod_binary.astype(bool)), cmap='gray', interpolation = 'nearest')
    axes[3].set_title('Multiscale Product')
    plt.show()


# Example Usage:
process_actin_fibers_correct(cy5_image, edge_method='gaussian')
# process_actin_fibers_correct(cy5_image, edge_method='mexican_hat')

## Canny Edge Detection

1. Noise reduction: with Gaussian or edge preserving smoothing
2. Finding the intensity gradient: Sobel filter (1st order derivative) in x and y directions. Magnitude of gradient gives edge strength. Direction tells edge orientation.
3. Non-maximum suppression: thin edges by keeping only the local maxima in the gradient direction. For example, if the gradient direction of a pixel is vertical. The algorithm compares pixel values above and below it. If it is the maximum, it is kept.
4. Hysteresis thresholding: determines if the thinned edges are real or not with two thresholds `minVal` and `maxVal`. Any edge pixel above `maxVal` is a strong edge. Any pixel between min and max are weak edges. Weak edges are only kept if they are connected to a strong edge, either directly or through other weak edges

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def canny_edge_detection_on_actin(image: np.ndarray, blur_kernel_size: int = 5, sigma: float = 1.4, low_multiplier: float = 0.8, high_multiplier: float = 1.2):
    """
    Loads a large image, performs Canny edge detection with automatic thresholding,
    and displays the result.

    Args:
        image_path (str): The path to the input image file.
        blur_kernel_size (int): The size of the Gaussian blur kernel (must be an odd number).
        sigma (float): The standard deviation for the Gaussian blur.
    """
    # --- 1. Load the Image in Grayscale ---
    # Canny edge detection works on single-channel images.
    # cv2.IMREAD_GRAYSCALE is the most direct way to load it as such.
    gray_image = image

    # --- 2. Preprocessing: Reduce Noise with a Gaussian Blur ---
    # This step is essential to prevent the Canny algorithm from detecting false edges
    # caused by noise in the microscope image.
    blurred_image = cv2.GaussianBlur(gray_image, (blur_kernel_size, blur_kernel_size), sigma)
    print("Applied Gaussian blur to reduce noise.")

    # --- 3. Automatic Threshold Calculation ---
    # We can automatically determine good thresholds by using the median of the
    # pixel intensities in the blurred image. This is more robust than hardcoding values.
    median_intensity = np.median(blurred_image)
    lower_threshold = int(max(0, low_multiplier * median_intensity))
    upper_threshold = int(min(255, high_multiplier * median_intensity))
    print(f"Automatically determined thresholds: Lower={lower_threshold}, Upper={upper_threshold}")

    # --- 4. Perform Canny Edge Detection ---
    # The algorithm finds edges where the intensity gradient is highest.
    # - Gradients above `upper_threshold` are definitely edges.
    # - Gradients below `lower_threshold` are definitely not edges.
    # - Gradients between the two are only considered edges if they are connected
    #   to a "strong" edge (one above the upper threshold).
    edges = cv2.Canny(blurred_image, lower_threshold, upper_threshold)
    print("Canny edge detection complete.")

    # --- 5. Visualize the Results ---
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))

    axes[0].imshow(gray_image, cmap='gray')
    axes[0].set_title('Original Actin Image')
    axes[0].axis('off')

    axes[1].imshow(blurred_image, cmap='gray')
    axes[1].set_title('Blurred Actin Image')
    axes[1].axis('off')
    
    print(edges.dtype)
    axes[2].imshow(invert_image_colors(edges), cmap='gray',interpolation='nearest')
    axes[2].set_title('Canny Edge Detection Result (Inverted)')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    
    return edges

canny_edges = canny_edge_detection_on_actin(cy5_image,
                              blur_kernel_size=1,
                              sigma = 3,
                              low_multiplier = 0.7,
                              high_multiplier = 1.3)

## Laplacian of Gaussian Filter

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def apply_log_filter(
    image: np.ndarray,
    sigma: float = 3.0,
    ksize: int = 0,
    laplacian_ksize: int = 3,
    ddepth: int = cv2.CV_64F,
    figsize: tuple = (16, 8),
    cmap: str = 'RdBu_r'
):
    """
    Applies a Laplacian of Gaussian (LoG) filter to an image with tunable parameters
    and displays the original versus the filtered result.

    The LoG filter is useful for blob and edge detection. It finds areas of
    rapid change (edges) in an image.

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale or color.
        sigma (float): The standard deviation for the Gaussian kernel. This controls
                       the scale of the features to be detected. A larger sigma
                       detects larger blobs/edges.
        ksize (int): The kernel size for the Gaussian blur. If set to 0, the size is
                     calculated automatically from sigma. Must be an odd number.
        laplacian_ksize (int): The size of the kernel for the Laplacian filter
                               (e.g., 1, 3, 5). A larger kernel is less sensitive
                               to noise. Must be an odd number.
        ddepth (int): The desired depth of the destination image for the Laplacian output
                      (e.g., cv2.CV_64F, cv2.CV_16S). Using a signed float type like
                      CV_64F is recommended to avoid information loss.
        figsize (tuple): A tuple specifying the width and height of the visualization figure.
        cmap (str): The colormap to use for displaying the LoG filtered image.
    """
    # --- 1. Preprocessing: Convert to Grayscale ---
    # The LoG filter operates on single-channel images.
    if image.ndim == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image.copy() # Assume it's already grayscale

    # --- 2. Apply Gaussian Blur ---
    # This step smooths the image and reduces noise, which is crucial for
    # preventing the Laplacian from detecting false edges.
    if ksize > 0 and ksize % 2 == 0:
        ksize += 1 # Ensure kernel size is odd
        print(f"Warning: Gaussian kernel size must be odd. Adjusting to {ksize}.")
    
    blur_k_size = (ksize, ksize) if ksize > 0 else (0, 0)
    blurred_image = cv2.GaussianBlur(gray_image, blur_k_size, sigmaX=sigma, sigmaY=sigma)

    # --- 3. Apply Laplacian Filter ---
    # We use a configurable data type (ddepth) to capture the full range
    # of positive and negative values from the second derivative.
    if laplacian_ksize % 2 == 0:
        laplacian_ksize += 1 # Ensure kernel size is odd
        print(f"Warning: Laplacian kernel size must be odd. Adjusting to {laplacian_ksize}.")
        
    log_filtered_image = cv2.Laplacian(blurred_image, ddepth, ksize=laplacian_ksize)

    print("Laplacian of Gaussian filter applied successfully.")
    print(f"Filtered image value range: [{log_filtered_image.min():.2f}, {log_filtered_image.max():.2f}]")

    # --- 4. Visualize the Results ---
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Display the original grayscale image
    axes[0].imshow(gray_image, cmap='gray')
    axes[0].set_title('Original Grayscale Image')
    axes[0].axis('off')

    # Display the LoG filtered image
    # A diverging colormap is used to visualize both positive (light) and
    # negative (dark) values, with zero being in the middle.
    # Zero-crossings in this map correspond to edges in the original image.
    im = axes[1].imshow(log_filtered_image, cmap=cmap)
    axes[1].set_title(f'Laplacian of Gaussian (Sigma={sigma}, K_size={laplacian_ksize})')
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1], orientation='vertical', fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
    
    return log_filtered_image


log_filtered_image = apply_log_filter(cy5_container.merge(), 
                 sigma=5, # gaussian stdev 
                 ksize=5, # gaussian kernel size
                 laplacian_ksize=9) # laplacian kernel size


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

# 1. Define the colors for the colormap
# The colors need to be defined as (R, G, B, Alpha)
# [0] = Transparent White/Black (0, 0, 0, 0)
# [1] = Grey (e.g., 0.5, 0.5, 0.5, 1.0) or Black (0, 0, 0, 1.0)

colors = [(0, 0, 0, 0),  # Transparent for 0 (False)
          (0, 0, 0, 1.0)] # Grey for 1 (True)

# 2. Create the custom colormap
custom_cmap = ListedColormap(colors)

boolean_matrix = (log_filtered_image > -10) & (log_filtered_image < 10)

plt.imshow(boolean_matrix, cmap=custom_cmap)
plt.title("Thresholded Filtered Image")
plt.colorbar(ticks=[0, 1], label='0=False, 1=True')
plt.show()

## Frangi filter

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import frangi

def detect_fibers_with_frangi(
    image: np.ndarray,
    scale_range: tuple = (1, 5),
    scale_step: float = 0.5,
    alpha: float = 0.5,
    beta: float = 0.5,
    gamma: float = 15,
    threshold: float = 0.1,
    figsize: tuple = (24, 8)
):
    """
    Detects and enhances fiber-like structures in an image using the Frangi filter.

    The Frangi filter is a Hessian-based filter that is excellent for detecting
    vessel-like or fiber-like structures of different sizes.

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale or color.
        scale_range (tuple): The range of sigmas (scales) to filter with. The filter
                             is applied for each scale in this range, and the maximum
                             response is taken. This should be set to match the
                             range of expected fiber widths in pixels.
        scale_step (float): The step size for iterating through the scale_range.
        alpha (float): Frangi filter parameter. Controls the sensitivity to blob-like
                       structures. Lower values reduce the response to blobs.
        beta (float): Frangi filter parameter. Controls the sensitivity to plate-like
                      structures (lines vs. planes).
        gamma (float): Frangi filter parameter. Controls the sensitivity to areas of
                       low contrast vs. high contrast. Higher values make the filter
                       more sensitive to faint structures.
        threshold (float): A value between 0 and 1 to binarize the Frangi filter's
                           output into a final mask.
        figsize (tuple): The size of the visualization figure.
    """
    # --- 1. Preprocessing: Convert to Grayscale and Normalize ---
    if image.ndim == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image.copy()

    # Normalize image to float [0, 1] for the filter
    if gray_image.dtype != np.float32 and gray_image.dtype != np.float64:
        gray_image = gray_image.astype(np.float32) / np.iinfo(gray_image.dtype).max

    # --- 2. Apply the Frangi Filter ---
    # The filter iterates through the sigmas in the scale_range to detect fibers
    # of different widths.
    print(f"Applying Frangi filter with scales from {scale_range[0]} to {scale_range[1]}...")
    frangi_filtered_image = frangi(
        gray_image,
        sigmas=np.arange(scale_range[0], scale_range[1], scale_step),
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        black_ridges=False # Use True if your fibers are dark on a light background
    )
    print("Frangi filter applied successfully.")

    # --- 3. Threshold the Vesselness Map to Get a Binary Mask ---
    binary_mask = (frangi_filtered_image > threshold).astype(np.uint8)
    print(f"Thresholded vesselness map at {threshold} to create binary mask.")

    # --- 4. Visualize the Results ---
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    axes[0].imshow(gray_image, cmap='gray')
    axes[0].set_title('Original Grayscale Image')
    axes[0].axis('off')

    im = axes[1].imshow(frangi_filtered_image, cmap='viridis')
    axes[1].set_title('Frangi Filter Output (Vesselness)')
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1], orientation='vertical', fraction=0.046, pad=0.04)

    axes[2].imshow(binary_mask, cmap='gray')
    axes[2].set_title(f'Final Binary Mask (Threshold > {threshold})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

detect_fibers_with_frangi(
    image = boolean_matrix.astype(np.uint8)*255,
    scale_range = (50, 60),
    scale_step = 0.5,
    alpha = 0.2,
    beta = 0.2,
    gamma = 50,
    threshold = 0.5,
    figsize = (24, 8)
)


## Hough Transform

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def detect_lines_with_hough(
    image: np.ndarray,
    blur_ksize: int = 5,
    canny_threshold1: int = 50,
    canny_threshold2: int = 150,
    hough_rho: float = 1,
    hough_theta: float = np.pi / 180,
    hough_threshold: int = 50,
    min_line_length: int = 50,
    max_line_gap: int = 10,
    figsize: tuple = (12, 12)
):
    """
    Detects line segments in an image using the Probabilistic Hough Transform.

    This function is well-suited for identifying straight or nearly-straight fibers.
    It first finds edges using the Canny algorithm and then uses those edges to
    vote for lines in the Hough space.

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale or color.
        blur_ksize (int): The kernel size for the initial Gaussian blur to reduce noise.
                          Must be an odd number.
        canny_threshold1 (int): The lower threshold for the Canny edge detector.
        canny_threshold2 (int): The upper threshold for the Canny edge detector.
        hough_rho (float): The distance resolution of the accumulator in pixels. A value of 1 is typical.
        hough_theta (float): The angle resolution of the accumulator in radians. np.pi/180
                             (1 degree) is typical.
        hough_threshold (int): The minimum number of votes (intersections in Hough grid)
                               a line needs to be detected. Lower this to detect more, fainter lines.
        min_line_length (int): The minimum length of a line in pixels. Shorter segments are discarded.
                               Increase this to filter out small, noisy lines.
        max_line_gap (int): The maximum allowed gap in pixels between segments on the same
                            line to link them. Increase this to connect dashed or broken fibers.
        figsize (tuple): The size of the visualization figure.
    """
    # --- 1. Preprocessing: Convert to Grayscale and Blur ---
    if image.ndim == 3 and image.shape[2] == 3:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray_image = image.copy()

    if blur_ksize > 0:
        if blur_ksize % 2 == 0:
            blur_ksize += 1
        blurred_image = cv2.GaussianBlur(gray_image, (blur_ksize, blur_ksize), 0)
    else:
        blurred_image = gray_image

    # --- 2. Canny Edge Detection ---
    edges = cv2.Canny(blurred_image, canny_threshold1, canny_threshold2)

    # --- 3. Probabilistic Hough Line Transform ---
    lines = cv2.HoughLinesP(
        edges,
        rho=hough_rho,
        theta=hough_theta,
        threshold=hough_threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )

    # --- 4. Visualize the Results ---
    # Create an image to draw the lines on
    lines_image = np.zeros_like(image)
    overlay_image = image.copy() if image.ndim == 3 else cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    if lines is not None:
        print(f"Detected {len(lines)} line segments.")
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw lines on the blank image (in green)
            cv2.line(lines_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Draw lines on the overlay image (in red)
            cv2.line(overlay_image, (x1, y1), (x2, y2), (255, 0, 0), 2)
    else:
        print("No lines were detected with the current parameters.")

    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, sharey=True)
    axes = axes.flatten()

    axes[0].imshow(gray_image, cmap='gray')
    axes[0].set_title('Original Grayscale Image')

    axes[1].imshow(edges, cmap='gray')
    axes[1].set_title('Canny Edges')

    axes[2].imshow(lines_image)
    axes[2].set_title('Detected Lines (Hough Transform)')

    axes[3].imshow(overlay_image)
    axes[3].set_title('Lines Overlaid on Original')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# --- Run the Line Detection Function ---
detect_lines_with_hough(
    cy5_image,
    canny_threshold1=50,
    canny_threshold2=200,
    hough_threshold=40,
    min_line_length=100,
    max_line_gap=20
)


In [ ]:
def detect_lines_with_hough(
    edges: np.ndarray,
    hough_rho: float = 1,
    hough_theta: float = np.pi / 180,
    hough_threshold: int = 50,
    min_line_length: int = 50,
    max_line_gap: int = 10,
    figsize: tuple = (12, 12)
):
    """
    Detects line segments in an image using the Probabilistic Hough Transform.

    This function is well-suited for identifying straight or nearly-straight fibers.
    It first finds edges using the Canny algorithm and then uses those edges to
    vote for lines in the Hough space.

    Args:
        image (np.ndarray): The input image matrix. Can be grayscale or color.
        blur_ksize (int): The kernel size for the initial Gaussian blur to reduce noise.
                          Must be an odd number.
        canny_threshold1 (int): The lower threshold for the Canny edge detector.
        canny_threshold2 (int): The upper threshold for the Canny edge detector.
        hough_rho (float): The distance resolution of the accumulator in pixels. A value of 1 is typical.
        hough_theta (float): The angle resolution of the accumulator in radians. np.pi/180
                             (1 degree) is typical.
        hough_threshold (int): The minimum number of votes (intersections in Hough grid)
                               a line needs to be detected. Lower this to detect more, fainter lines.
        min_line_length (int): The minimum length of a line in pixels. Shorter segments are discarded.
                               Increase this to filter out small, noisy lines.
        max_line_gap (int): The maximum allowed gap in pixels between segments on the same
                            line to link them. Increase this to connect dashed or broken fibers.
        figsize (tuple): The size of the visualization figure.
    """
    # --- 1. Preprocessing: Convert to Grayscale and Blur ---
    # --- 3. Probabilistic Hough Line Transform ---
    lines = cv2.HoughLinesP(
        edges,
        rho=hough_rho,
        theta=hough_theta,
        threshold=hough_threshold,
        minLineLength=min_line_length,
        maxLineGap=max_line_gap
    )

    # --- 4. Visualize the Results ---
    # Create an image to draw the lines on
    lines_image = np.zeros_like(edges)
    overlay_image = edges.copy() if edges.ndim == 3 else cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    if lines is not None:
        print(f"Detected {len(lines)} line segments.")
        for line in lines:
            x1, y1, x2, y2 = line[0]
            # Draw lines on the blank image (in green)
            cv2.line(lines_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # Draw lines on the overlay image (in red)
            cv2.line(overlay_image, (x1, y1), (x2, y2), (255, 0, 0), 2)
    else:
        print("No lines were detected with the current parameters.")

    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=True, sharey=True)
    axes = axes.flatten()

    axes[0].imshow(edges, cmap='gray')
    axes[0].set_title('Original Grayscale Image')

    axes[1].imshow(edges, cmap='gray')
    axes[1].set_title('Canny Edges')

    axes[2].imshow(lines_image)
    axes[2].set_title('Detected Lines (Hough Transform)')

    axes[3].imshow(overlay_image)
    axes[3].set_title('Lines Overlaid on Original')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    
    return lines

# --- Run the Line Detection Function ---
_ = detect_lines_with_hough(
    boolean_matrix.astype(np.uint8)*255,
    hough_threshold=200,
    min_line_length=500,
    max_line_gap=10
)


## Derivative of Gaussian

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from skimage import data

def apply_derivative_of_gaussian(image: np.ndarray, sigma: float = 1.0):
    """
    Applies a Derivative of Gaussian (DoG) filter to an image.

    This function first smooths the image with a Gaussian filter and then
    computes its gradient in the x and y directions. It returns the
    individual derivatives and the combined gradient magnitude.

    Args:
        image (np.ndarray): The input image as a 2D NumPy array.
        sigma (float): The standard deviation (sigma) of the Gaussian kernel.
                       A larger sigma detects larger-scale edges.

    Returns:
        Tuple[np.ndarray, np.ndarray, np.ndarray]: A tuple containing:
            - grad_magnitude (np.ndarray): The gradient magnitude, highlighting edges.
            - grad_x (np.ndarray): The derivative in the x-direction (vertical edges).
            - grad_y (np.ndarray): The derivative in the y-direction (horizontal edges).
    """
    # Ensure the image is float for derivative calculations
    img_float = image.astype(np.float32)

    # Calculate the derivative in the y-direction (axis 0)
    # The 'order' parameter specifies the derivative order for each axis.
    # order=(1, 0) means 1st derivative along axis 0, 0th derivative along axis 1.
    grad_y = gaussian_filter(img_float, sigma=sigma, order=(1, 0), mode='nearest')

    # Calculate the derivative in the x-direction (axis 1)
    # order=(0, 1) means 0th derivative along axis 0, 1st derivative along axis 1.
    grad_x = gaussian_filter(img_float, sigma=sigma, order=(0, 1), mode='nearest')

    # Calculate the gradient magnitude
    grad_magnitude = np.sqrt(grad_x**2 + grad_y**2)

    return grad_magnitude, grad_x, grad_y

# --- Example Usage ---

# 1. Load a sample image
original_image = cy5_image

# 2. Set the sigma for the Gaussian filter
sigma_value = 1.5

# 3. Apply the Derivative of Gaussian filter
magnitude, dx, dy = apply_derivative_of_gaussian(original_image, sigma=sigma_value)

# 4. Display the results
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(original_image, cmap='gray')
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(magnitude, cmap='gray')
axes[1].set_title(f"Gradient Magnitude (σ={sigma_value})")
axes[1].axis('off')

axes[2].imshow(dx, cmap='gray')
axes[2].set_title("X-Derivative (Vertical Edges)")
axes[2].axis('off')

axes[3].imshow(dy, cmap='gray')
axes[3].set_title("Y-Derivative (Horizontal Edges)")
axes[3].axis('off')

plt.tight_layout()
plt.show()


## Marr-Hildreth

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

def marr_hildreth_custom(
    image: np.ndarray,
    sigma: float = 1.0,
    gaussian_ksize: int = 0,
    laplacian_ksize: int = 3,
    threshold: float = 0.01
):
    """
    Applies a customizable Marr-Hildreth edge detector to an image.

    This implementation separates the Gaussian blur and Laplacian filter steps,
    allowing for individual control over their kernel sizes.

    Args:
        image (np.ndarray): 
            The input 2D image as a NumPy array.
        sigma (float): 
            The standard deviation for the Gaussian kernel. Controls the scale of edges.
        gaussian_ksize (int): 
            The kernel size for the Gaussian blur. If 0, it's calculated from sigma. Must be odd.
        laplacian_ksize (int): 
            The kernel size for the Laplacian filter. Must be odd.
        threshold (float): 
            A sensitivity threshold for zero-crossings, as a fraction of the max
            LoG value, to filter out weak edges.

    Returns:
        np.ndarray: A binary image where 1s represent detected edges.
    """
    # --- 1. Apply Laplacian of Gaussian (LoG) with custom kernels ---
    
    # Ensure image is float for calculations
    img_float = image.astype(np.float32)

    # Apply Gaussian Blur
    if gaussian_ksize > 0 and gaussian_ksize % 2 == 0:
        gaussian_ksize += 1  # Ensure kernel size is odd
    blur_k_size = (gaussian_ksize, gaussian_ksize) if gaussian_ksize > 0 else (0, 0)
    blurred_image = cv2.GaussianBlur(img_float, blur_k_size, sigmaX=sigma, sigmaY=sigma)

    # Apply Laplacian Filter
    if laplacian_ksize % 2 == 0:
        laplacian_ksize += 1 # Ensure kernel size is odd
    # Use CV_64F for ddepth to capture the full range of positive/negative values
    log_filtered = cv2.Laplacian(blurred_image, cv2.CV_64F, ksize=laplacian_ksize)

    # --- 2. Find Zero-Crossings ---
    # This logic remains the same as your original marr_hildreth_filter.
    padded_log = np.pad(log_filtered, 1, 'edge')
    positive_pixels = (padded_log > 0)
    
    # Check for neighbors with opposite signs in all 8 directions
    min_neighbors = np.min(np.stack([
        positive_pixels[1:-1, :-2],   # Left
        positive_pixels[1:-1, 2:],    # Right
        positive_pixels[:-2, 1:-1],   # Top
        positive_pixels[2:, 1:-1],    # Bottom
        positive_pixels[:-2, :-2],    # Top-Left
        positive_pixels[:-2, 2:],     # Top-Right
        positive_pixels[2:, :-2],     # Bottom-Left
        positive_pixels[2:, 2:]       # Bottom-Right
    ]), axis=0)
    
    # zero_crossings = (is the central pixel positive?) & (is at least one neighbor NOT positive?)
    zero_crossings = positive_pixels[1:-1, 1:-1] & ~min_neighbors

    # --- 3. Apply Threshold to Filter Weak Edges ---
    local_max_diff = np.max(np.stack([
        np.abs(padded_log[1:-1, :-2] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[1:-1, 2:] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[:-2, 1:-1] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[2:, 1:-1] - padded_log[1:-1, 1:-1])
    ]), axis=0)

    threshold_value = threshold * np.max(np.abs(log_filtered))
    edges = zero_crossings & (local_max_diff > threshold_value)

    return padded_log, edges.astype(np.uint8)

# --- Example Usage ---

# 1. Load a sample image
original_image = cy5_image

# 2. Set parameters for the custom filter
sigma_value = 5
gaussian_kernel_size = 5  # Explicitly set Gaussian kernel size
laplacian_kernel_size = 9 # Use a larger Laplacian kernel
edge_threshold = 0.05

# 3. Apply the custom Marr-Hildreth filter
log_out, edge_map = marr_hildreth_custom(
    original_image,
    sigma=sigma_value,
    gaussian_ksize=gaussian_kernel_size,
    laplacian_ksize=laplacian_kernel_size,
    threshold=edge_threshold
)

# 4. Display the results
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(original_image, cmap='gray')
axes[0].set_title("Original Image")
axes[0].axis('off')

title = (f"Marr-Hildreth Edges\n"
         f"σ={sigma_value}, G_ksize={gaussian_kernel_size}, "
         f"L_ksize={laplacian_kernel_size}, thresh={edge_threshold}")
axes[1].imshow(edge_map, cmap='gray')
axes[1].set_title(title)
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from skimage.util import random_noise

def enhanced_marr_hildreth(
    image: np.ndarray,
    laplacian_ksize: int = 3,
    threshold: float = 0.01,
    smoothing_method: str = 'gaussian',
    smoothing_params: dict = None
):
    """
    Applies an enhanced Marr-Hildreth edge detector with customizable pre-smoothing.

    This algorithm finds edges by:
    1. Applying a user-selected smoothing filter (Gaussian, Median, or Bilateral) to reduce noise.
    2. Applying a Laplacian filter to find regions of high second-order derivative.
    3. Detecting zero-crossings in the filtered image, which correspond to edges.
    4. Thresholding the zero-crossings based on their local gradient to remove weak, noisy edges.

    Args:
        image (np.ndarray):
            The input 2D image as a NumPy array.
        laplacian_ksize (int):
            The kernel size for the Laplacian filter (e.g., 1, 3, 5). Must be odd.
        threshold (float):
            A sensitivity threshold (0 to 1) for zero-crossings. It is a fraction of the
            max LoG value, used to filter out weak edges.
        smoothing_method (str):
            The pre-smoothing method: 'gaussian', 'median', or 'bilateral'.
        smoothing_params (dict):
            A dictionary of parameters for the chosen smoothing method.
            - For 'gaussian': {'sigma': float, 'ksize': int}
            - For 'median': {'ksize': int}
            - For 'bilateral': {'d': int, 'sigmaColor': float, 'sigmaSpace': float}

    Returns:
        np.ndarray: A binary image where 1s represent detected edges.
    """
    # --- Step 1: Pre-smoothing to Reduce Noise ---
    # This is a critical first step. The Laplacian operator is very sensitive to noise,
    # so we first smooth the image. The choice of filter can significantly impact the result.
    if smoothing_params is None:
        smoothing_params = {}
    
    img_float = image.astype(np.float32)
    smoothed_image = None

    if smoothing_method.lower() == 'gaussian':
        sigma = smoothing_params.get('sigma', 1.0)
        ksize = smoothing_params.get('ksize', 0)
        if ksize > 0 and ksize % 2 == 0: ksize += 1
        blur_k_size = (ksize, ksize) if ksize > 0 else (0, 0)
        smoothed_image = cv2.GaussianBlur(img_float, blur_k_size, sigmaX=sigma, sigmaY=sigma)

    elif smoothing_method.lower() == 'median':
        # Median filter is excellent for salt-and-pepper noise.
        # It requires an 8-bit input image.
        ksize = smoothing_params.get('ksize', 5)
        if ksize % 2 == 0: ksize += 1
        smoothed_image = cv2.medianBlur(image.astype(np.uint8), ksize).astype(np.float32)

    elif smoothing_method.lower() == 'bilateral':
        # Bilateral filter smooths flat regions while preserving sharp edges.
        d = smoothing_params.get('d', 9)
        sigma_color = smoothing_params.get('sigmaColor', 75)
        sigma_space = smoothing_params.get('sigmaSpace', 75)
        smoothed_image = cv2.bilateralFilter(img_float, d, sigma_color, sigma_space)
    
    else:
        raise ValueError("Unsupported smoothing_method. Choose 'gaussian', 'median', or 'bilateral'.")

    # --- Step 2: Apply Laplacian Filter ---
    # The Laplacian is a second-order derivative operator that finds areas of rapid
    # intensity change. We use CV_32F to capture both positive and negative values.
    if laplacian_ksize % 2 == 0:
        laplacian_ksize += 1
    log_filtered = cv2.Laplacian(smoothed_image, cv2.CV_32F, ksize=laplacian_ksize)

    # --- Step 3: Detect Zero-Crossings ---
    # Edges are located where the LoG output changes sign. This is found by identifying
    # positive pixels that have at least one non-positive (zero or negative) neighbor.
    padded_log = np.pad(log_filtered, 1, 'edge')
    positive_pixels = (padded_log > 0)
    
    # Check all 8 neighbors. If any neighbor is non-positive, the central pixel is on an edge.
    min_neighbors = np.min(np.stack([
        positive_pixels[1:-1, :-2], positive_pixels[1:-1, 2:],    # Left, Right
        positive_pixels[:-2, 1:-1], positive_pixels[2:, 1:-1],    # Top, Bottom
        positive_pixels[:-2, :-2],  positive_pixels[:-2, 2:],     # Diagonals
        positive_pixels[2:, :-2],   positive_pixels[2:, 2:]
    ]), axis=0)
    
    # A zero-crossing occurs where the central pixel is positive AND not all its neighbors are positive.
    zero_crossings = positive_pixels[1:-1, 1:-1] & ~min_neighbors

    # --- Step 4: Thresholding by Strength ---
    # To remove weak edges caused by noise, we measure the "strength" of each zero-crossing.
    # The strength is the maximum absolute difference between the pixel and its neighbors.
    local_max_diff = np.max(np.stack([
        np.abs(padded_log[1:-1, :-2] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[1:-1, 2:] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[:-2, 1:-1] - padded_log[1:-1, 1:-1]),
        np.abs(padded_log[2:, 1:-1] - padded_log[1:-1, 1:-1])
    ]), axis=0)

    # Set a threshold as a fraction of the maximum LoG response.
    threshold_value = threshold * np.max(np.abs(log_filtered))
    
    # The final edge map contains only pixels that are both a zero-crossing AND strong enough.
    edges = zero_crossings & (local_max_diff > threshold_value)

    return edges.astype(np.uint8)

# --- Example Usage ---

# 1. Load a sample image and add salt-and-pepper noise
original_image = cy5_image
noisy_image = (random_noise(original_image, mode='s&p', amount=0.05) * 255).astype(np.uint8)
noisy_image = original_image

# 2. Apply the enhanced Marr-Hildreth filter with different smoothing methods
# Gaussian smoothing (traditional LoG)
edges_gaussian = enhanced_marr_hildreth(
    noisy_image, laplacian_ksize=9, threshold=0.04,
    smoothing_method='gaussian', smoothing_params={'sigma': 5, 'ksize': 5}
)

# Median smoothing (good for salt-and-pepper noise)
edges_median = enhanced_marr_hildreth(
    noisy_image, laplacian_ksize=9, threshold=0.04,
    smoothing_method='median', smoothing_params={'ksize': 5}
)

# Bilateral smoothing (preserves edges well)
edges_bilateral = enhanced_marr_hildreth(
    noisy_image, laplacian_ksize=9, threshold=0.04,
    smoothing_method='bilateral', smoothing_params={'d': 9, 'sigmaColor': 75, 'sigmaSpace': 75}
)

# With original image and median smoothing
edges_original = enhanced_marr_hildreth(
    original_image, laplacian_ksize=9, threshold=0.04,
    smoothing_method='median', smoothing_params={'ksize': 5}
)

# 3. Display the results for comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(original_image, cmap='gray')
axes[0, 0].set_title("Original Image")
axes[0, 0].axis('off')

axes[0, 1].imshow(noisy_image, cmap='gray')
axes[0, 1].set_title("Image with Salt & Pepper Noise")
axes[0, 1].axis('off')

axes[0, 2].imshow(edges_gaussian, cmap='gray')
axes[0, 2].set_title("Marr-Hildreth with Gaussian Smoothing")
axes[0, 2].axis('off')

axes[1, 0].imshow(edges_median, cmap='gray')
axes[1, 0].set_title("Marr-Hildreth with Median Smoothing")
axes[1, 0].axis('off')

axes[1, 1].imshow(edges_bilateral, cmap='gray')
axes[1, 1].set_title("Marr-Hildreth with Bilateral Smoothing")
axes[1, 1].axis('off')

axes[1, 2].imshow(edges_original, cmap='gray')
axes[1, 2].set_title("Marr-Hildreth with Original Image and Median Smoothing")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## Difference of Gaussian

In [ ]:
import cv2
import numpy as np

def apply_difference_of_gaussians(img: np.ndarray, sigma1: float = 1.5, sigma2: float = 2.4, threshold: float = None) -> np.ndarray:
    """
    Applies a Difference of Gaussians (DoG) filter to an image.
    
    Args:
        img (np.ndarray): Input 2D image.
        sigma1 (float): Sigma for the first Gaussian kernel.
        sigma2 (float): Sigma for the second Gaussian kernel.
        threshold (float): Threshold value for binarization. If None, Otsu's method is used.
        
    Returns:
        np.ndarray: The binary edge map (0s and 1s).
    """
    # Apply edge preserving smoothing
    img = apply_edge_preserving_smoothing(img, method="median")
    
    # Apply two Gaussian blurs
    # Kernel size is set to (0, 0) so OpenCV calculates it automatically based on sigma
    g1 = cv2.GaussianBlur(img, (0, 0), sigma1)
    g2 = cv2.GaussianBlur(img, (0, 0), sigma2)
    
    # Subtract the results (DoG)
    # Convert to float64 to prevent overflow/underflow during subtraction of integer types
    dog = g1.astype(np.float64) - g2.astype(np.float64)
    
    # Take the absolute value to capture the magnitude of the difference (edges/blobs)
    dog = np.abs(dog)

    # Min-Max Normalization to match the input image's data type range
    # This ensures the output is viewable and compatible with functions like invert_image_colors
    if np.issubdtype(img.dtype, np.integer):
        info = np.iinfo(img.dtype)
        min_val = dog.min()
        max_val = dog.max()
        
        if max_val > min_val:
            # Scale to [0, info.max] (e.g., 0-255 or 0-65535)
            dog = (dog - min_val) / (max_val - min_val) * info.max
        else:
            # Handle constant image case (avoid division by zero)
            dog = np.zeros_like(dog)
            
        dog = dog.astype(img.dtype)
        
    # Thresholding to return binary edge map (0s and 1s)
    if threshold is None:
        # Use Otsu's thresholding
        # OpenCV Otsu implementation requires 8-bit input image
        if dog.dtype != np.uint8:
            # Map to 0-255 for Otsu calculation
            dog_for_otsu = cv2.normalize(dog, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        else:
            dog_for_otsu = dog
            
        # Apply Otsu's thresholding; maxval=1 gives 0s and 1s
        _, binary_edges = cv2.threshold(dog_for_otsu, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        binary_edges = (dog > threshold).astype(np.uint8)
        
    return binary_edges

sigma1 = 3
sigma2 = 5
dog_image = apply_difference_of_gaussians(cy5_image, sigma1, sigma2)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(cy5_image, cmap='gray')
axes[0].set_title("Original Image")
axes[0].axis('off')

title = (f"Difference of Gaussian")
axes[1].imshow(invert_image_colors(dog_image), cmap='gray')
axes[1].set_title(title)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from image_processing_tools.util.visualize import add_gradient_arrows_to_ax

def apply_difference_of_gaussians(img: np.ndarray, sigma1: float = 1.5, sigma2: float = 2.4, threshold: float = None) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Applies a Difference of Gaussians (DoG) filter to an image.
    
    Args:
        img (np.ndarray): Input 2D image.
        sigma1 (float): Sigma for the first Gaussian kernel.
        sigma2 (float): Sigma for the second Gaussian kernel.
        threshold (float): Threshold value for binarization. If None, Otsu's method is used.
        
    Returns:
        A tuple containing:
        - median_denoised (np.ndarray): The image after median filtering.
        - g1 (np.ndarray): The result of the first Gaussian blur.
        - dog_magnitude (np.ndarray): The continuous DoG result before thresholding.
        - binary_edges (np.ndarray): The binary edge map (0s and 1s).
    """
    # Apply edge preserving smoothing
    median_denoised = apply_edge_preserving_smoothing(img, method="median")
    
    # Apply two Gaussian blurs
    g1 = cv2.GaussianBlur(median_denoised, (0, 0), sigma1)
    g2 = cv2.GaussianBlur(median_denoised, (0, 0), sigma2)
    
    # Subtract the results (DoG)
    dog = g1.astype(np.float64) - g2.astype(np.float64)
    
    # Take the absolute value to capture the magnitude
    dog_magnitude = np.abs(dog)

    # Min-Max Normalization
    if np.issubdtype(img.dtype, np.integer):
        info = np.iinfo(img.dtype)
        min_val, max_val = dog_magnitude.min(), dog_magnitude.max()
        
        if max_val > min_val:
            dog_magnitude = ((dog_magnitude - min_val) / (max_val - min_val) * info.max)
        else:
            dog_magnitude = np.zeros_like(dog_magnitude)
            
        dog_magnitude = dog_magnitude.astype(img.dtype)
        
    # Thresholding
    if threshold is None:
        if dog_magnitude.dtype != np.uint8:
            dog_for_otsu = cv2.normalize(dog_magnitude, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        else:
            dog_for_otsu = dog_magnitude
        _, binary_edges = cv2.threshold(dog_for_otsu, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        binary_edges = (dog_magnitude > threshold).astype(np.uint8)
        
    return median_denoised, g1, dog_magnitude, binary_edges

# --- Main Execution ---
# Assuming 'cy5_image' is your input image defined elsewhere

sigma1 = 3
sigma2 = 5
median_denoised, g1_blur, dog_continuous, dog_binary = apply_difference_of_gaussians(cy5_image, sigma1, sigma2)

# Calculate gradients from one of the blurred images (g1) for arrow orientation
gx = cv2.Sobel(g1_blur.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(g1_blur.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Original Image
axes[0].imshow(cy5_image, cmap='gray')
axes[0].set_title("Original Image")

# Plot 2: Median Denoised Image
axes[1].imshow(median_denoised, cmap='gray')
axes[1].set_title("Median Denoised with Gradient Arrow Overlay")
# add_gradient_arrows_to_ax(axes[1], gx, gy, step=100, rotate_90=True, alpha=0.5,arrow_scale=3)

# Plot 3: Difference of Gaussians with Gradient Arrows
axes[2].imshow(invert_image_colors(dog_binary), cmap='gray',interpolation = "nearest")
axes[2].set_title(f"DoG (σ={sigma1},{sigma2})")


for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()


### Difference of Gaussian vs. Mexican hat

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gaussian(x, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-x**2 / (2 * sigma**2))

def mexican_hat_log(x, sigma):
    # Standard formula for Laplacian of Gaussian
    return (1 / (np.sqrt(2 * np.pi) * sigma**3)) * (1 - (x**2 / sigma**2)) * np.exp(-x**2 / (2 * sigma**2))

# Parameters
x = np.linspace(-5, 5, 1000)
sigma_narrow = 1.0
sigma_wide = 1.6 # k = 1.6 is the optimum approximation

# DoG = Narrow Blur - Wide Blur
dog = gaussian(x, sigma_narrow) - gaussian(x, sigma_wide)

# Mexican Hat (LoG)
log = mexican_hat_log(x, sigma_narrow)

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(x, dog, label='DoG (Narrow - Wide)', color='blue', linewidth=2)
plt.plot(x, log * 0.6, '--', label='Mexican Hat (LoG)', color='red', linewidth=2)

plt.axhline(0, color='black', lw=1)
plt.title("The 'Mexican Hat' Shape: DoG vs LoG")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()